In [2]:
from pathlib import Path
from itertools import product
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
from arch import arch_model
from sklearn.metrics import roc_auc_score
from statsmodels.tools.sm_exceptions import PerfectSeparationError

ALPHAS = [0.01, 0.03, 0.05]
HORIZONS = [1, 5, 10, 21]
MODELS = ["GARCH22_FHS", "Rolling_HS"]
WINDOW_MODE = "observations"
WINDOW_OBS = 1260
WINDOW_YEARS = 5
MIN_WINDOW_OBS = 200
END_DATE = "2025-10-30"
SCOPES = {
    "FullSample": ("2007-10-01", END_DATE),
    "COVID": ("2020-01-01", "2020-12-31"),
    "Inflation": ("2022-01-01", "2022-12-31"),
}
SCOPE_HAC_LAGS = {"FullSample": 10, "COVID": 5, "Inflation": 5}
OUT = Path("btc_cross_asset_daily_return")


def load_returns():
    raw = yf.download(
        "BTC-USD",
        start="2000-01-01",
        end=(pd.Timestamp(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
        auto_adjust=False,
        progress=False,
    )
    close = raw["Close"]

    if isinstance(close, pd.DataFrame):
        close = close["BTC-USD"] if "BTC-USD" in close.columns else close.iloc[:, 0]

    close = close.dropna().sort_index()
    close.index = pd.DatetimeIndex(close.index).tz_localize(None).normalize()

    if close.empty:
        raise ValueError("No BTC prices downloaded.")
    if close.index.has_duplicates or (close <= 0).any():
        raise ValueError("Duplicate dates or nonpositive closing prices.")
    if not close.index.to_series().diff().dropna().eq(pd.Timedelta(days=1)).all():
        raise ValueError("Missing BTC calendar dates.")

    returns = np.log(close).diff().dropna().rename("Return")

    if not np.isfinite(returns).all():
        raise ValueError("Nonfinite returns.")

    return returns


def historical_laws(window):
    mu = float(window.mean())
    sigma = float(window.std(ddof=0))

    if not np.isfinite(sigma) or sigma <= 0:
        raise ValueError("Invalid historical volatility.")

    residuals = (window.to_numpy(dtype=float) - mu) / sigma
    quantiles = np.quantile(residuals, ALPHAS, method="inverted_cdf")

    return {
        h: (
            mu + sigma * quantiles,
            mu,
            sigma,
            "Daily empirical distribution with constant within-window moments",
        )
        for h in HORIZONS
    }


def garch_laws(window):
    model = arch_model(
        100 * window,
        mean="Constant",
        vol="GARCH",
        p=2,
        o=0,
        q=2,
        dist="normal",
        rescale=False,
    )
    result = model.fit(
        disp="off",
        show_warning=False,
        options={"maxiter": 1000},
    )

    if result.convergence_flag != 0 or not np.isfinite(result.params).all():
        raise ValueError("GARCH optimizer failed to converge.")

    residuals = np.asarray(result.std_resid, dtype=float)
    residuals = residuals[np.isfinite(residuals)]

    if len(residuals) < 2:
        raise ValueError("Insufficient valid residuals.")

    residual_sd = residuals.std(ddof=0)

    if not np.isfinite(residual_sd) or residual_sd <= 0:
        raise ValueError("Invalid FHS residual distribution.")

    residuals = (residuals - residuals.mean()) / residual_sd
    quantiles = np.quantile(residuals, ALPHAS, method="inverted_cdf")

    prediction = result.forecast(
        horizon=max(HORIZONS),
        method="analytic",
        reindex=False,
    )

    laws = {}

    for h in HORIZONS:
        mu = float(prediction.mean.iloc[-1, h - 1]) / 100
        variance = float(prediction.variance.iloc[-1, h - 1]) / 10000

        if not np.isfinite(mu) or not np.isfinite(variance) or variance <= 0:
            raise ValueError(f"Invalid daily-return forecast at lead time {h}.")

        sigma = np.sqrt(variance)
        laws[h] = (
            mu + sigma * quantiles,
            mu,
            sigma,
            "Daily FHS distribution using h-step-ahead GARCH volatility",
        )

    return laws


def make_forecasts(returns):
    returns = returns.dropna().sort_index()

    if WINDOW_MODE not in {"observations", "calendar_years"}:
        raise ValueError("Invalid WINDOW_MODE.")

    values = returns.to_numpy(dtype=float)
    dates = returns.index
    start = pd.Timestamp(SCOPES["FullSample"][0])
    end = pd.Timestamp(END_DATE)
    rows = []
    completed = 0

    for i in range(len(returns) - 1):
        origin = dates[i]

        eligible_horizons = [
            h for h in HORIZONS
            if i + h < len(returns) and start <= dates[i + h] <= end
        ]

        if not eligible_horizons:
            continue

        if WINDOW_MODE == "observations":
            if i + 1 < WINDOW_OBS:
                continue
            window = returns.iloc[i - WINDOW_OBS + 1:i + 1]
        else:
            cutoff = origin - pd.DateOffset(years=WINDOW_YEARS)
            if dates[0] > cutoff:
                continue
            window = returns.loc[(dates > cutoff) & (dates <= origin)]

        if len(window) < MIN_WINDOW_OBS:
            continue

        for model_name in MODELS:
            error = ""

            try:
                laws = (
                    historical_laws(window)
                    if model_name == "Rolling_HS"
                    else garch_laws(window)
                )
            except Exception as exc:
                error = f"{type(exc).__name__}: {exc}"
                laws = None

            for h in eligible_horizons:
                evaluation = dates[i + h]
                actual = float(values[i + h])

                if laws is None:
                    quantiles = np.full(len(ALPHAS), np.nan)
                    mean_h = sd_h = np.nan
                    basis = "Unavailable"
                else:
                    quantiles, mean_h, sd_h, basis = laws[h]

                for j, alpha in enumerate(ALPHAS):
                    model_var = -float(quantiles[j])
                    cantelli = -mean_h + sd_h * np.sqrt((1 - alpha) / alpha)
                    status = "ok" if laws is not None else "forecast_error"

                    if status == "ok" and (
                        not np.isfinite(sd_h) or sd_h <= 0
                    ):
                        status = "invalid_forecast_volatility"

                    if status == "ok" and (
                        not np.isfinite(cantelli) or cantelli <= 0
                    ):
                        status = "nonpositive_or_invalid_cantelli"

                    if status == "ok" and (
                        not np.isfinite(model_var) or model_var < 0
                    ):
                        status = "negative_or_invalid_model_loss_var"

                    ver = model_var / cantelli if status == "ok" else np.nan

                    if status == "ok" and ver > 1 + 1e-10:
                        status = "cantelli_ordering_violation"
                        ver = np.nan

                    rows.append({
                        "Model": model_name,
                        "Alpha": alpha,
                        "ForecastHorizon": h,
                        "ForecastOrigin": origin,
                        "EvaluationDate": evaluation,
                        "WindowStart": window.index[0],
                        "WindowObs": len(window),
                        "ForecastMean": mean_h,
                        "ForecastSD": sd_h,
                        "ModelLossVaR": model_var,
                        "CantelliLossVaR": cantelli,
                        "VaR_Ratio": ver,
                        "ActualReturn": actual,
                        "Breach": (
                            int(-actual > model_var)
                            if np.isfinite(model_var) else np.nan
                        ),
                        "ForecastStatus": status,
                        "MomentBasis": basis,
                        "Details": error,
                    })

        completed += 1

        if completed % 100 == 0:
            print(
                f"Completed {completed} origins; latest: {origin.date()}",
                flush=True,
            )

    return pd.DataFrame(rows)


def select_scope(forecasts, scope):
    start, end = map(pd.Timestamp, SCOPES[scope])
    return forecasts.loc[
        forecasts["EvaluationDate"].between(start, end)
    ].copy()


def fit_logit(data, lags):
    if data.empty:
        return None, "No valid observations"

    if data["Breach"].nunique() < 2:
        return None, (
            "No breaches"
            if data["Breach"].sum() == 0
            else "All observations are breaches"
        )

    if data["VaR_Ratio"].nunique() < 2:
        return None, "Constant VER"

    X = sm.add_constant(data[["VaR_Ratio"]], has_constant="add")

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            result = sm.Logit(data["Breach"].astype(int), X).fit(
                disp=False,
                maxiter=300,
                cov_type="HAC",
                cov_kwds={
                    "maxlags": int(lags),
                    "kernel": "bartlett",
                    "use_correction": False,
                },
            )

        if any("separation" in str(w.message).lower() for w in caught):
            return None, "Separation"

        if not result.mle_retvals.get("converged", False):
            return None, "Non-convergence"

        if not np.isfinite(
            np.r_[result.params, result.bse, result.pvalues]
        ).all():
            return None, "Nonfinite estimates"

        return result, "Estimated"

    except PerfectSeparationError:
        return None, "Separation"
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def summarize(forecasts, scope):
    rows = []

    for alpha, h in product(ALPHAS, HORIZONS):
        raw = forecasts.loc[
            forecasts["Alpha"].eq(alpha)
            & forecasts["ForecastHorizon"].eq(h)
        ]
        sub = raw.loc[
            raw["ForecastStatus"].eq("ok")
        ].sort_values("EvaluationDate")

        lags = SCOPE_HAC_LAGS[scope]
        result, status = fit_logit(sub, lags)

        row = {
            "Sample": scope,
            "Tail probability": alpha,
            "Forecast horizon (days)": h,
            "Sample start": sub["EvaluationDate"].min(),
            "Sample end": sub["EvaluationDate"].max(),
            "Observations": len(sub),
            "Breaches": int(sub["Breach"].sum()),
            "VER coefficient": np.nan,
            "HAC p-value": np.nan,
            "In-sample AUC": np.nan,
            "HAC lags": lags,
            "Excluded forecasts": len(raw) - len(sub),
            "Estimation status": status,
        }

        if result is not None:
            X = sm.add_constant(sub[["VaR_Ratio"]], has_constant="add")
            probabilities = result.predict(X)

            row.update({
                "VER coefficient": float(result.params["VaR_Ratio"]),
                "HAC p-value": float(result.pvalues["VaR_Ratio"]),
                "In-sample AUC": float(
                    roc_auc_score(sub["Breach"], probabilities)
                ),
            })

        rows.append(row)

    return pd.DataFrame(rows)


def format_p_value(value):
    if pd.isna(value):
        return "N/A"
    return "< 0.001" if value < 0.001 else f"{value:.3f}"


def export_summary(summary, path):
    report = summary.copy()
    report["Sample"] = report["Sample"].replace({
        "FullSample": "Full sample",
        "COVID": "COVID-19",
        "Inflation": "Inflation",
    })
    report["Tail probability"] = report["Tail probability"].map(
        lambda value: f"{100 * value:.0f}%"
    )
    report["HAC p-value"] = report["HAC p-value"].map(format_p_value)

    for column in ["Sample start", "Sample end"]:
        report[column] = pd.to_datetime(
            report[column]
        ).dt.strftime("%Y-%m-%d")

    report.to_csv(
        path,
        index=False,
        na_rep="N/A",
        float_format="%.6g",
    )


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    returns = load_returns()
    forecasts = make_forecasts(returns)

    if forecasts.empty:
        raise RuntimeError("No eligible forecasts.")

    output_names = {
        "GARCH22_FHS": "Summary_GARCH22_FHS.csv",
        "Rolling_HS": "Summary_Rolling_HS.csv",
    }

    for model_name in MODELS:
        model_forecasts = forecasts.loc[
            forecasts["Model"].eq(model_name)
        ].copy()

        summary = pd.concat(
            [
                summarize(select_scope(model_forecasts, scope), scope)
                for scope in SCOPES
            ],
            ignore_index=True,
        )

        if len(summary) != len(SCOPES) * len(ALPHAS) * len(HORIZONS):
            raise RuntimeError("Missing prespecified summary cells.")

        export_summary(summary, OUT / output_names[model_name])

    forecasts.rename(columns={
        "Alpha": "Tail probability",
        "ForecastHorizon": "Forecast horizon (days)",
        "ForecastOrigin": "Forecast origin",
        "EvaluationDate": "Evaluation date",
        "WindowStart": "Estimation window start",
        "WindowObs": "Estimation window observations",
        "ForecastMean": "Forecast daily return mean",
        "ForecastSD": "Forecast daily return standard deviation",
        "ModelLossVaR": "Model loss VaR",
        "CantelliLossVaR": "Cantelli loss VaR",
        "VaR_Ratio": "VER",
        "ActualReturn": "Realised daily log return",
        "Breach": "VaR breach",
        "ForecastStatus": "Forecast status",
        "MomentBasis": "Forecast distribution",
    }).to_csv(
        OUT / "VER_forecast_cross_asset.csv",
        index=False,
        na_rep="N/A",
    )

    print(f"Three CSV files saved in {OUT.resolve()}")


if __name__ == "__main__":
    main()

Completed 100 origins; latest: 2018-06-07
Completed 200 origins; latest: 2018-09-15
Completed 300 origins; latest: 2018-12-24
Completed 400 origins; latest: 2019-04-03
Completed 500 origins; latest: 2019-07-12
Completed 600 origins; latest: 2019-10-20
Completed 700 origins; latest: 2020-01-28
Completed 800 origins; latest: 2020-05-07
Completed 900 origins; latest: 2020-08-15
Completed 1000 origins; latest: 2020-11-23
Completed 1100 origins; latest: 2021-03-03
Completed 1200 origins; latest: 2021-06-11
Completed 1300 origins; latest: 2021-09-19
Completed 1400 origins; latest: 2021-12-28
Completed 1500 origins; latest: 2022-04-07
Completed 1600 origins; latest: 2022-07-16
Completed 1700 origins; latest: 2022-10-24
Completed 1800 origins; latest: 2023-02-01
Completed 1900 origins; latest: 2023-05-12
Completed 2000 origins; latest: 2023-08-20
Completed 2100 origins; latest: 2023-11-28
Completed 2200 origins; latest: 2024-03-07
Completed 2300 origins; latest: 2024-06-15
Completed 2400 origi